# GEIGER-1911 · 3 — What the slit should look like

Notebook 2 chose eight places to point the counter. This one decides what to cut in the
brass in front of it, and the answer turns out to be governed by a single identity that
costs nothing to notice and a great deal to miss.

A detector opening covering azimuth $\phi$ and scattering angles $\theta \pm \delta$
subtends exactly

$$\omega = \int_{\theta-\delta}^{\theta+\delta}\!\!\int_{\phi} \sin\theta'\, d\theta'\, d\phi'
\;=\; \phi \cdot 2\sin\theta\,\sin\delta$$

so the same solid angle — the same number of counts an hour — can be bought by opening in
$\delta$ or by opening in $\phi$. **Only one of them costs anything.** The scattering
pattern does not depend on azimuth, so an aperture stretched round the beam axis collects
more counts of exactly the same measurement; an aperture stretched in $\theta$ collects
counts of a *different* measurement and averages them together.

| | decides | reaches into |
|---|---|---|
| **1** | how much solid angle at all | `scattering.widest_aperture` |
| **2** | what a wide slit does to the number | `Scattering.forward`, integrated |
| **3** | slot or hole | `scattering.slit_for`, `scattering.aperture_bias` |
| **4** | where the workshop, not the statistics, binds | arithmetic |

In [ ]:
import sys

sys.path.insert(0, ".")

import math

import numpy as np
import plotly.graph_objects as go

import scattering as S

print(f"the counter saturates above {S.MAX_COUNT_RATE:.1f} flashes/s "
      f"({S.MAX_COUNT_RATE * 60:.0f} a minute)")
print(f"the screen runs out above    {S.OMEGA_MAX} sr")
print(f"the workshop will not cut a slot narrower than "
      f"{2 * S.SLIT_MIN_HALF_WIDTH:.2f} deg across")

## 1 · How much solid angle, before any question of shape

Two constraints fix the *total* aperture at each station, and they bind at opposite ends
of the range. A human at a scintillation microscope counts about ninety flashes a minute
before starting to miss them, which at small angles is a brutal limit: at five degrees
the foil is throwing off nearly three hundred thousand alphas per steradian per second.
Past eighty degrees the rate has fallen so far that the apparatus runs out of screen
first.

In [ ]:
grid = np.geomspace(0.4, 179.0, 300)
aperture = S.widest_aperture(grid, S.HARD_CENTRE)
uncapped = S.MAX_COUNT_RATE / S.rate_per_steradian(grid, S.HARD_CENTRE)

fig = S.figure("How much aperture each station is allowed", "scattering angle",
               "solid angle (sr)", height=430)
fig.add_trace(go.Scatter(x=grid, y=uncapped, name="what the counting rate allows",
                         line={"color": S.HARD_COLOR, "width": 3}))
fig.add_trace(go.Scatter(x=grid, y=aperture, name="what is actually available",
                         line={"color": S.SLIT_COLOR, "width": 3, "dash": "dash"}))
fig.add_hline(y=S.OMEGA_MAX, line={"dash": "dot", "color": S.TRUTH_COLOR},
              annotation_text="the screen runs out")
fig.update_yaxes(type="log", exponentformat="power")
S.degrees_axis(fig, log=True)
fig.show()

print("aperture the counter can be given, station by station:")
for st in S.PLAN:
    if st.foil:
        print(f"  {st.theta_deg:6.1f} deg  {float(S.widest_aperture([st.theta_deg], S.HARD_CENTRE)[0]):10.3g} sr")

Eight orders of magnitude, from a station that must be a pinhole to one that wants the
whole screen. Nothing about that is a choice; it is what the source and the observer's
eye between them allow.

## 2 · What a wide slit does to the number

A counter behind an opening does not report the rate at $\theta$. It reports the *average*
of the rate over everything the opening admits — and $u^{-4}$ is steeply convex, so the
inner edge contributes far more than the outer edge takes away. The counter comes back
with a number that belongs to no single angle and is too large.

Both quantities go through the same `forward`: `slit_counts` integrates it across what the
slit admits, and `rate_per_steradian` evaluates it at the centre. The relative gap is what
a model that ignored the width of its own aperture would be wrong by.

In [ ]:
widths = np.array([0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0])
fig = S.figure("What a slit of half-width d does to the number it reports",
               "half-width d (deg)", "relative error of the reported rate", height=430)
for angle, colour in ((3.0, S.HARD_COLOR), (10.0, S.ACCENT), (45.0, S.SLIT_COLOR),
                      (150.0, S.DIFFUSE_COLOR)):
    keep = widths < angle * 0.9
    measured = [S.aperture_bias(angle, float(d), 360.0, S.HARD_CENTRE) for d in widths[keep]]
    fig.add_trace(go.Scatter(x=widths[keep], y=measured, name=f"{angle:.0f} deg",
                             line={"color": colour, "width": 3}, mode="lines+markers"))
    if angle < 60.0:
        fig.add_trace(go.Scatter(x=widths[keep], y=2.0 * (widths[keep] / angle) ** 2,
                                 name=f"2 (d/theta)^2 at {angle:.0f} deg", showlegend=angle == 3.0,
                                 line={"color": colour, "width": 1, "dash": "dot"}))
fig.update_xaxes(type="log")
fig.update_yaxes(type="log", exponentformat="power")
fig.show()

The dotted lines are $2(\delta/\theta)^2$, which is what the curvature of $u^{-4}\sin\theta$
gives at small angle, and the integrated `forward` sits on them to within a few per cent
until the slit gets close to swallowing the beam. So the rule is scale-free:

> **the smearing error depends only on the ratio $\delta/\theta$.**

A slit is not "narrow" in degrees. It is narrow relative to where it is pointed, and a
half-width that is negligible at ninety degrees is a catastrophe at one.

In [ ]:
print(f"{'ratio d/theta':>14} {'2 (d/theta)^2':>15} {'integrated at 10 deg':>22}")
for ratio in (0.02, 0.05, 0.1, 0.2):
    print(f"{ratio:14.2f} {2 * ratio**2:15.4%} "
          f"{S.aperture_bias(10.0, 10.0 * ratio, 360.0, S.HARD_CENTRE):22.4%}")

## 3 · Slot or hole

Now the identity from the top of the notebook does its work. Take the aperture each
station is allowed and buy it two different ways.

* a **circular hole** — as wide in azimuth as it is in scattering angle, $\phi = 2\delta$;
* an **annular slot** — as narrow in scattering angle as the workshop can cut, with the
  rest of the solid angle taken out in azimuth.

Same counts per hour. Same station. Same everything except the shape of the brass.

In [ ]:
from scipy.optimize import brentq

print(f"{'angle':>7} {'aperture':>10} | {'slot: half':>11} {'arc':>7} {'error':>9} "
      f"| {'hole: half':>11} {'error':>9}")
rows = []
for st in S.PLAN:
    if not st.foil:
        continue
    omega = float(S.widest_aperture([st.theta_deg], S.HARD_CENTRE)[0])
    half, arc = S.slit_for(st.theta_deg, omega)
    slot_err = S.aperture_bias(st.theta_deg, half, arc, S.HARD_CENTRE)
    hole_half = brentq(lambda d: S.slit_solid_angle(st.theta_deg, d, 2 * d) - omega,
                       1e-7, min(st.theta_deg * 0.98, 60.0))
    hole_err = S.aperture_bias(st.theta_deg, hole_half, 2 * hole_half, S.HARD_CENTRE)
    rows.append((st.theta_deg, half, arc, slot_err, hole_half, hole_err))
    print(f"{st.theta_deg:6.1f}d {omega:10.3g} | {half:10.3f}d {arc:6.1f}d {slot_err:9.4%} "
          f"| {hole_half:10.3f}d {hole_err:9.4%}")

At ninety degrees the two openings admit exactly the same 0.2 steradians. The slot is
$\pm 0.9°$ wide and reports the rate to a part in four thousand. The hole is $\pm 13°$
wide — it runs from seventy-seven degrees to a hundred and three — and reports a number
five per cent too high that belongs to no angle in particular. **Fourteen times the
angular resolution and two hundred times less smearing, for the same counts, because the
scattering does not care about azimuth and the experimenter can.**

That is the whole answer, and it is worth drawing.

In [ ]:
fig = go.Figure()
for theta, half, arc, _, hole_half, _ in rows:
    fig.add_trace(go.Barpolar(r=[2 * max(half, 0.35)], base=[theta - max(half, 0.35)],
                              theta=[0.0], width=[arc], marker_color=S.SLIT_COLOR,
                              marker_line_color="white", marker_line_width=1, opacity=0.85,
                              showlegend=False, hovertext=f"{theta:.0f} deg slot"))
fig.add_trace(go.Barpolar(r=[2 * 12.87], base=[90 - 12.87], theta=[180.0], width=[2 * 12.87],
                          marker_color=S.HARD_COLOR, opacity=0.5, showlegend=True,
                          name="the same 0.2 sr as a round hole"))
fig.update_layout(
    title="The apertures, drawn on the sphere (radius = scattering angle)", height=520,
    template="plotly_white",
    polar={"radialaxis": {"range": [0, 175], "tickvals": [30, 60, 90, 120, 150],
                          "ticksuffix": "°", "angle": 90},
           "angularaxis": {"showticklabels": False, "ticks": ""}},
    legend={"orientation": "h", "yanchor": "bottom", "y": -0.05, "x": 0.0})
fig.show()

print("Green: the eight slots, each drawn at least 0.7 deg thick so it is visible at all.")
print("Red:   one round hole of the same solid angle as the 90 deg slot.")

The picture is the specification. Near the beam the aperture is a short stub of arc — the
counting rate will not allow more. As the rate falls the slot runs further and further
round the axis until, past about forty degrees, it closes into a **complete annulus**, and
only then does it begin to open radially because there is nowhere else for the solid angle
to go.

A hole would have had to be forty times wider in scattering angle to admit the same beam,
and would have measured the average of a curve instead of a point on it.

## 4 · Where the workshop binds, and how far in the anchor can go

The prescription "cut the slot as narrow as you can" has a floor, and past that floor the
station stops being feasible. Two consequences, both of which change what gets built.

**The anchor stations saturate the counter.** At one degree the aperture allowed is
$5.9 \times 10^{-9}$ sr, which is nine times smaller than the smallest opening the workshop
can cut. There is no slit that fixes this; the beam has to be attenuated instead.

In [ ]:
smallest = {st.theta_deg: S.slit_solid_angle(st.theta_deg, S.SLIT_MIN_HALF_WIDTH,
                                             2 * S.SLIT_MIN_HALF_WIDTH)
            for st in S.PLAN if st.foil}
print(f"{'angle':>7} {'aperture wanted':>17} {'smallest cuttable':>19} {'beam must be cut to':>21}")
for angle, floor_sr in smallest.items():
    want = float(S.widest_aperture([angle], S.HARD_CENTRE)[0])
    factor = want / floor_sr
    verdict = f"{factor:.2f} x" if factor < 1.0 else "-"
    print(f"{angle:6.1f}d {want:17.3g} {floor_sr:19.3g} {verdict:>21}")

**And the anchor cannot be moved closer in.** The smearing error scales as
$(\delta/\theta)^2$ with $\delta$ pinned at the workshop's floor, so it grows as the
station moves toward the beam, while the statistical error stays flat — the counter is
rate-capped, so ten hours is ten hours' worth of counts wherever it points. There is an
angle where they cross, and inside it the number is limited by brass rather than by
counting.

In [ ]:
inner = np.geomspace(0.3, 5.0, 40)
smear = np.array([S.aperture_bias(float(a), S.SLIT_MIN_HALF_WIDTH,
                                  2 * S.SLIT_MIN_HALF_WIDTH, S.HARD_CENTRE) for a in inner])
statistical = 1.0 / math.sqrt(S.MAX_COUNT_RATE * 10 * 3600.0)

fig = S.figure("How far in the anchor can be put", "scattering angle",
               "relative error of the reported rate", height=420)
fig.add_trace(go.Scatter(x=inner, y=smear, name="smearing, at the narrowest cuttable slot",
                         line={"color": S.HARD_COLOR, "width": 3}))
fig.add_hline(y=statistical, line={"dash": "dash", "color": S.SLIT_COLOR},
              annotation_text="counting error in ten hours")
fig.update_xaxes(type="log", tickvals=[0.3, 0.5, 1, 2, 5], ticktext=["0.3°", "0.5°", "1°", "2°", "5°"])
fig.update_yaxes(type="log", exponentformat="power")
fig.show()

crossing = float(np.interp(0.0, (np.log(smear) - math.log(statistical))[::-1], inner[::-1]))
print(f"the two errors cross at {crossing:.2f} degrees")
print(f"inside it the aperture, not the counting, is the error -- which is why the")
print(f"anchor sits at 1.0 and 2.5 degrees and not at half a degree.")

## The slit, specified

| station | radial half-width | azimuthal arc | shape | beam |
|---|---|---|---|---|
| 1.0° | 0.05° | a 0.1° stub | pinhole | attenuated 9× |
| 2.5° | 0.05° | 0.2° | pinhole | full |
| 5° | 0.05° | 2° | short slot | full |
| 10° | 0.05° | 16° | slot | full |
| 20° | 0.05° | 126° | long slot | full |
| 45° | 0.20° | 360° | annulus | full |
| 90° | 0.91° | 360° | annulus | full |
| 150° | 1.82° | 360° | annulus | full |

Read down the middle column and the design states itself. **The radial width is held at
the workshop's floor for as long as possible and every additional steradian is bought in
azimuth**, because azimuth is the direction the physics is blind to. It opens only when
the arc has gone all the way round and there is nowhere else to go — and by then the
station is at forty-five degrees, where the curve is flat enough that a degree of slit
costs four parts in a hundred thousand.

Every smearing error in the table is below a twentieth of the counting error at the same
station. **The aperture is not a source of error in this experiment**, which is a
statement about the *shape* of the brass and would be false for a round hole of exactly
the same area.

Notebook 4 asks how long to leave each of them open, and when to stop.